In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

In [54]:
file = r"C:\Users\Admin\Desktop\Amazon\amazon_sales_data_2025.csv"
df = pd.read_csv(file)

In [55]:
print("Original data shape: ", df.shape)
print(df.head())
print(df.info())
print(df.describe(include="all"))

Original data shape:  (250, 11)
  Order ID       Date        Product     Category  Price  Quantity  \
0  ORD0001   14-03-25  Running Shoes     Footwear     60         3   
1  ORD0002   20-03-25     Headphones  Electronics    100         4   
2  ORD0003   15-02-25  Running Shoes     Footwear     60         2   
3  ORD0004   19-02-25  Running Shoes     Footwear     60         3   
4  ORD0005  10/3/2025     Smartwatch  Electronics    150         3   

   Total Sales  Customer Name Customer Location Payment Method     Status  
0          180     Emma Clark          New York     Debit Card  Cancelled  
1          400  Emily Johnson     San Francisco     Debit Card    Pending  
2          120       John Doe            Denver     Amazon Pay  Cancelled  
3          180  Olivia Wilson            Dallas    Credit Card    Pending  
4          450     Emma Clark          New York     Debit Card    Pending  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total

In [56]:
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(" ", "_")
              .str.replace("-", "_")
             )

In [57]:
print(df.columns.tolist())

['order_id', 'date', 'product', 'category', 'price', 'quantity', 'total_sales', 'customer_name', 'customer_location', 'payment_method', 'status']


In [60]:
date_col = "date"
initial_count = len(df)
df = df[~df[date_col].astype(str).str.contains("########", na=False)]
removed_count = initial_count - len(df)
print(f"Removed {removed_count} rows with '####' dates")
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
initial_count_2 = len(df)
df = df.dropna(subset=[date_col])
removed_count_2 = initial_count_2 - len(df)
print(f"Removed {removed_count_2} additional rows with invalid dates")
df[date_col] = df[date_col].dt.strftime("%d-%m-%Y")
print("Date column sample after cleaning:")
print(df[date_col].head())

Removed 0 rows with '####' dates
Removed 0 additional rows with invalid dates
Date column sample after cleaning:
0    14-03-2025
1    20-03-2025
2    15-02-2025
3    19-02-2025
4    03-10-2025
Name: date, dtype: object


C:\Users\Admin\AppData\Local\Temp\ipykernel_2812\319000027.py:6: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[date_col] = pd.to_datetime(df[date_col], errors='coerce')


In [46]:
df[date_col] = df[date_col].apply(fix_date_cell)

print("Date column after initial fixing:")
print(df[date_col].head(10))

Date column after initial fixing:
0    2025-03-2014
1    2025-03-2020
2    2025-02-2015
3    2025-02-2019
4    2025-03-2010
5    2025-03-2014
6    2025-03-2018
7    2025-03-2002
8    2025-03-2008
9    2025-03-2012
Name: date, dtype: object


In [61]:
df[date_col] = pd.to_datetime(df[date_col], format='%d-%m-%Y', errors='coerce')

In [62]:
df[date_col] = df[date_col].dt.strftime("%d-%m-%Y")

In [63]:
print("Date column sample after final cleaning:")
print(df[date_col].head(10))

Date column sample after final cleaning:
0    14-03-2025
1    20-03-2025
2    15-02-2025
3    19-02-2025
4    03-10-2025
5    14-03-2025
6    18-03-2025
7    03-02-2025
8    03-08-2025
9    03-12-2025
Name: date, dtype: object


In [64]:
def remove_outliers_zscore(df, threshold=3):
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    z_scores = np.abs((df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std())
    return df[(z_scores < threshold).all(axis=1)]

df = remove_outliers_zscore(df)

In [65]:
clean_file = df.to_csv(r"C:\Users\Admin\Desktop\Amazon\cleaned_amazon_data.csv", index=False)